# __Lecture 09: Practical analyses in Python__

----
### **_Be sure to pull the latest notebook!!_**

#### __Announcement:__ Homework 4 is assigned today and is due November 5.

#### __On Tuesday (lecture 08), we:__
- motivated pattern recognition
- read and dissected regular expressions with the `re` module
- wrote __test cases__ for a pattern, used them to find bugs and fix them

#### __Today we will:__
- warm up with a few regular expression predictions
- use the `biopython` module to read a real FASTQ file and work with sequences
- apply regular expressions to real data!

__Homework 4 extends today's analysis__, so keep this notebook handy.

###  
----

## __Review:__ regular expression predictions

The pattern cheat sheet is in [lecture08.ipynb](../lecture08/lecture08.ipynb) - keep it open.

For each cell below, __predict the output__, then run it. 

In [1]:
import re

# 1. Which part of the strain name does each pattern pull out?
strain = "A/Puerto Rico/8/1934 (H1N1)"

print(re.compile(r"\d{4}").search(strain))
print(re.compile(r"^[AB]").search(strain))
print(re.compile(r"(?P<city>[A-Za-z ]+)/\d").search(strain).group("city"))

<re.Match object; span=(16, 20), match='1934'>
<re.Match object; span=(0, 1), match='A'>
Puerto Rico


In [2]:
# 2. Two of these find a match and one does not. Which one, and why?
read = "ACGTACGTAGGCGGCCGCTTTTAAAACCCCGGGG"

print(re.compile(r"AGGCGGCCGC(?P<bc>[ACGT]{16})").search(read))
print(re.compile(r"^AGGCGGCCGC(?P<bc>[ACGT]{16})").search(read))
print(re.compile(r"AGGCGGCCGC(?P<bc>[ACGT]{16})$").search(read))

<re.Match object; span=(8, 34), match='AGGCGGCCGCTTTTAAAACCCCGGGG'>
None
<re.Match object; span=(8, 34), match='AGGCGGCCGCTTTTAAAACCCCGGGG'>


----
# __New module: `biopython`__

The [Biopython](https://biopython.org/) package is a long-standing collection of tools for
computational biology. We will use two small pieces of it:

- [`Bio.SeqIO`](https://biopython.org/wiki/SeqIO) - reading and writing sequence files (FASTA, FASTQ,
  GenBank, and ~30 other formats). 
- [`Bio.Seq`](https://biopython.org/wiki/Seq) - a sequence object that knows it is a biological
  sequence, with methods like `reverse_complement()`, `transcribe()`, and `translate()`.

#### The FASTQ format

Our data file, `barcodes_R1.fastq`, is in FASTQ format: __four lines per read__.

    @HISEQ:621:HMJGNBCX2:1:1101:1797:2150 1:N:0:ATGTCA   <- read name (the Illumina header from Tuesday!)
    GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATT...           <- the sequence
    +                                                     <- a separator
    GGGGGGIIIGIIIIGIIIIGIIIIIIIGGIGIIIIIIII...           <- one quality character per base

That fourth line is the part people forget about, and we will come back to it.

In [3]:
import Bio.SeqIO
import Bio.Seq

# SeqIO.parse returns reads one at a time (lazily); list() pulls them all into memory
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

print(f"Found {len(seqreads)} sequencing reads.")

Found 10000 sequencing reads.


#### Reads come back as `SeqRecord` objects

A [SeqRecord](https://biopython.org/wiki/SeqRecord) bundles together everything the file knew
about one read: its name, its sequence, and its per-base quality scores.

In [16]:
read = seqreads[0]

print(type(read))
print(read)

<class 'Bio.SeqRecord.SeqRecord'>
ID: HISEQ:621:HMJGNBCX2:1:1101:1797:2150
Name: HISEQ:621:HMJGNBCX2:1:1101:1797:2150
Description: HISEQ:621:HMJGNBCX2:1:1101:1797:2150 1:N:0:ATGTCA
Number of features: 0
Per letter annotation for: phred_quality
Seq('GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTGA...AGA')


In [18]:
# the pieces you will actually use:
print("id:      ", read.id)
print("sequence:", read.seq)
print("length:  ", len(read.seq))
print("quality: ", read.letter_annotations["phred_quality"])

id:       HISEQ:621:HMJGNBCX2:1:1101:1797:2150
sequence: GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTGAAAGGGAGTTCTGCTCCATCAGGCCAAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAGCTGAAATTAATAATTTTGAAACCAGTTTTGTAAACGCAGCACTAAAATGAAGGCATGCAACGACGATGTTTATTGACACGGAATAGCAGA
length:   250
quality:  [38, 38, 38, 38, 38, 38, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 38, 38, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 38, 40, 40, 40, 38, 40, 38, 40, 40, 40, 38, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 40, 40, 40, 38, 40, 40, 40, 40, 40, 38, 38, 40, 40, 40, 40, 38, 38, 38, 38, 40, 38, 32, 38, 38, 40, 38, 38, 38, 38, 38, 40, 38, 40, 40, 40, 27, 32, 38, 38, 38, 40, 40, 38, 38, 40, 40, 38, 38, 40, 38, 40, 38, 38, 32, 38, 40, 38, 38, 40, 38, 38, 38, 38, 40, 40, 40, 40, 38, 32, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2

#### Sequences are `Seq` objects, not strings

A `Seq` behaves a lot like a string (you can index it, slice it, and take its length), but it also
has biology-aware methods. If you need to hand the sequence to something that expects a plain
string - including the `re` module - convert it with `str()`.

In [6]:
seq = seqreads[0].seq

print(type(seq))
print(seq[:30])

# converting to a string when we need one
seq_string = str(seq)
print(type(seq_string))

<class 'Bio.Seq.Seq'>
GCTTAAGTTATTTAGTGCGGCCGCCTATGG
<class 'str'>


In [7]:
# a few of the built-in Seq methods
short = Bio.Seq.Seq("ATGGCCATTGTAATGGGCCGC")

print("sequence:          ", short)
print("complement:        ", short.complement())
print("reverse complement:", short.reverse_complement())
print("transcribed (RNA): ", short.transcribe())
print("translated:        ", short.translate())

sequence:           ATGGCCATTGTAATGGGCCGC
complement:         TACCGGTAACATTACCCGGCG
reverse complement: GCGGCCCATTACAATGGCCAT
transcribed (RNA):  AUGGCCAUUGUAAUGGGCCGC
translated:         MAIVMGR


__Note how much work `reverse_complement()` is doing for you__ - compare it with the hand-written
`reverse_complement` function from lecture 6. 

#### Another note: `biopython` is a great default for reading sequence files and for small-to-medium analyses like this one. It is not always the fastest option. 

----
# __A real analysis: parsing barcodes from a sequencing run__

The reads in `barcodes_R1.fastq` come from a real sequencing run of influenza virus HA and NA
genes. Each molecule in the library was built like this:

    5'-[ end of HA or NA ]-AGGCGGCCGC-[ 16-nucleotide barcode ]-3'

The sequencer read these molecules __from the other end__, so what is actually in our file is the
reverse complement:

    5'-[ reverse complement of the barcode ]-GCGGCCGCCT-[ reverse complement of the end of HA/NA ]-3'

You can see this in the very first read: the first 16 bases are the (reverse complemented) barcode,
immediately followed by `GCGGCCGCCT`.

In [19]:
print(str(seqreads[0].seq)[:40])
#      |<----- barcode, reverse complemented ----->||<- GCGGCCGCCT ->||<- end of HA/NA sequence, reverse complemented ->|

GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTA


#### Our plan

1. __reverse complement__ each read, which puts it back in the orientation of the original
   molecule - and puts the barcode at the end, right after `AGGCGGCCGC`
2. __test__ whether it matches the expected design
3. __extract__ the barcode and count it

Steps 2 and 3 are exactly the scan/test/report pattern from Tuesday, with a 16-base barcode and a
10-base upstream sequence instead of the toy construct.

### __Step 0: say what "correct" means, before writing the pattern__

Just like on Tuesday, we write the requirements down first, in plain English. After reverse
complementing, a valid read should satisfy all of:

__Requirements:__
1. the read contains the fixed upstream sequence `AGGCGGCCGC`
2. the barcode is exactly __16__ bases long
3. every barcode base is one of `A`, `C`, `G`, `T`, `N` - and nothing else
4. the barcode is reported in upper case, in the forward orientation

Anything that fails one of these is not a valid barcode, and the parser should return `None` for it.



### __Step 1: write the parsing function__

The documentation is written for you. __Write the body of this function__, and then (in step 2)
write the tests that convince you it is right.

_Hint 1: this is the same shape as `parse_barcode_v2` from Tuesday._

_Hint 2: `seqread.reverse_complement()` does the orientation flip for you._

_Hint 3: `re` works on strings, not `Seq` objects - convert with `str()` first._

_Hint 4: use the input variables `upstream` and `bclen` to build your pattern._

In [ ]:
def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """
    # your code here...

    return None

### __Step 2: write test cases to check it__

Now turn the requirements from step 0 into tests, the same way as Tuesday: one
`(input, expected answer, label)` tuple per case, collected in a list that `run_tests` runs for you.

* a __positive__ case - a good read, and the barcode it should give back
* a __negative__ case - an input that violates __that one requirement__, for which `read_barcode`
  should return `None`

__Your task:__ add a test case for each of requirements 1-4 to the list below. One positive case and
one negative case are there to show the format.

__A note on orientation:__ the function receives a read in the orientation of the __file__, so any
test input you write has to be in that orientation too - ending with `GCGGCCGCCT`, not starting with
`AGGCGGCCGC`.

In [ ]:
# the same run_tests helper we wrote on Tuesday
def run_tests(parser_function, test_cases):
    """Run a parser against a list of test cases and report the results.

    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.

    Returns:
        The number of test cases that failed.
    """
    n_failed = 0

    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return n_failed

In [ ]:
test_cases = [
    # (read, expected result, label saying which requirement this checks)
    # positive: a clean read (R1-R4)
    ("TTTTTTTTTTTTTTTTGCGGCCGCCT", "AAAAAAAAAAAAAAAA", "positive: clean read"),
    # negative: the upstream constant sequence is corrupted (R1)
    ("TTTTTTTACGTTTTTTGCGGGCCT", None, "R1: upstream sequence is wrong"),
    # YOUR TEST CASES HERE
]

run_tests(read_barcode, test_cases)

### __Step 3: run it on all 10,000 reads__

Now the part that is only possible because we trust the function: apply it to the whole file and
count the barcodes. Keep track of the reads that __did not__ match - that count is a result, not
an error.

In [ ]:
barcode_counts = {}
unmatched = []

for read in seqreads:
    # your code here...

----
# __Wrapping up__

#### The workflow from these two lectures

1. __Describe the design in plain English__ before writing any code
2. __Write test cases__ that encode that description, including negatives
3. __Write the pattern__, and run it against the test cases
4. __Run analysis on real data__

#### Where this goes next

- __Homework 4__ extends today's analysis: the reads come from __either__ HA or NA, and you will
  work out which, using the sequence at the far end of the read. Almost all of the code you need is
  in this notebook.
- __Lecture 15__ (Introduction to sequencing data) picks up the sequencing side in much more depth.
- __Lecture 17__ (Immune repertoire sequencing) applies this same kind of parsing to a different
  biological problem.